***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 3 章：干涉测量中的位置天文学](3_0_introduction.ipynb)
    * 上一节：[第 3 章：干涉测量中的位置天文学](3_0_introduction.ipynb)
    * 下一节：[3.2 时角、地方恒星时与过中天](3_2_hour_angle.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


In [ ]:
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
HTML('../style/code_toggle.html')


## 3.1 赤道坐标、参考历元与星表位置<a id='pos:sec:eq'></a>

对干涉测量来说，赤道坐标系不是“诸多天文坐标中的一种可选坐标”，而是观测流程的起点。科学目标源、相位校准源、带通校准源和成像视场中心，最初通常都来自目录条目；这些目录位置几乎总是以赤经 $\alpha$ 和赤纬 $\delta$ 给出。因此，理解赤道坐标的意义，不只是知道两条角坐标如何定义，还要清楚它们在参考架、历元和实际观测中的角色。

干涉仪最终测量的是相位差和几何延迟。相位差对方向误差非常敏感：一个看似很小的角度误差，乘上很长的基线以后就可能变成不可忽略的相位误差。赤道坐标的精确定义，正是后续时角、指向、相位中心和延迟模型能够保持一致的基础。


### 从地理坐标到天球坐标

地理坐标用经度和纬度唯一标记地球表面的位置；赤道坐标则把同样的思路搬到天球上。只要观测时间尺度不太长，大多数射电源都可以近似看作位于无限远处，因此它们在天球上的方向可以用单位球面上的一点来表示。这就是*天球近似*。把地球赤道投影到天球上得到天赤道，把地球自转轴延伸到天球上得到北天极和南天极，于是便自然构成了赤道坐标的几何基础。


<a id='pos:fig:geo'></a><img src='figures/geo.png' width='58%'>

*图 3.1.1*：地理坐标中的经度与纬度。赤道坐标与其最直观的类比，是把这一套经纬度结构投影到天球上。


在赤道坐标中，赤纬 $\delta$ 由天赤道向南北量起，取值范围为 $[-90^\circ, +90^\circ]$；赤经 $\alpha$ 则沿天赤道从春分点向东量起。春分点是黄道与天赤道的交点之一，也是传统赤道坐标的零点方向。与地理经度常用角度不同，赤经通常记为小时、分钟和秒：全天 `24 h` 对应 $360^\circ$，也就是 $1 h = 15^\circ$。

若暂时忽略参考架细节，一个赤道坐标方向可以写成单位向量

$$
\mathbf{s}(\alpha,\delta)=
\begin{pmatrix}
\cos\delta\cos\alpha\\
\cos\delta\sin\alpha\\
\sin\delta
\end{pmatrix}.
$$

这个向量形式很有用，因为后面的许多量本质上都是向量点乘：天顶方向与源方向的点乘给出仰角，基线向量与源方向的点乘给出几何延迟，相位中心方向与邻近源方向的点乘则给出局部角距离。


<div class='warn'>
<b>记号提醒：</b> 赤经经常写成时角单位，赤纬写成角度单位，但在推导公式和写代码时最好统一转换到弧度。若把“1 小时”和“1 度”混用，坐标变换会出现整整 15 倍的尺度错误。
</div>


<a id='pos:fig:equatorial_coordinates'></a><img src='figures/equatorial.png' width='55%'>

*图 3.1.2*：赤道坐标 $\alpha$ 与 $\delta$ 的定义。赤经沿天赤道从春分点向东量起，赤纬从天赤道向北或向南量起。


### 赤道坐标在干涉观测中的角色

在真正的观测流程里，赤道坐标至少承担三种任务。它首先是源目录的语言：科学目标和校准源通常以 `RA/Dec` 进入观测准备软件。它也是相位中心的语言：成像坐标、相位旋转和局部方向余弦都必须相对于某个赤道坐标定义的中心方向建立。它还是调度几何的入口：地方恒星时、时角、仰角和观测窗口都要从赤经赤纬出发计算。

因此，赤道坐标不是一套“静态展示天图”的符号系统，而是后续全部台站几何与成像几何的源头。第 3.2 节会把赤经接到地方恒星时，第 3.3 节会把赤经赤纬转成方位角和高度角，第 3.4 节则会以相位中心为原点重新组织视场内的方向。


### 参考架、历元与目录位置

在专业文献中，讨论一个天源位置时至少要分清参考架、历元和位置类型。参考架或参考系统，例如 `ICRS`、`FK5`、`FK4`，规定坐标轴相对于惯性空间如何定义。历元或纪元，例如 `J2000.0`，说明坐标值对应的参考时刻或传统赤道零点约定。位置类型则进一步区分目录位置、瞬时视位置、地心位置和台站顶心位置等。

对射电干涉课程而言，最常见的近似说法是“该源的坐标为 `J2000` 赤经赤纬”。严格地说，这通常意味着：目录值被引用到现代惯性参考框架的某种近似表述，并采用 `J2000.0` 作为常用标签。老文献中常见的 `B1950` 属于更早的坐标系统；如果不做转换，就不能与现代 `ICRS/J2000` 目录直接混用。对于太阳系天体、近邻恒星或高精度脉冲星，位置还会随时间明显变化，此时自行、视差、光行差和星历模型也必须进入计算。


<div class='advice'>
<b>本书约定：</b> 后续如果没有特别声明，我们把目录坐标理解为现代惯性参考架下的 `ICRS/J2000` 近似表述。真正用于望远镜指向和高精度延迟模型的岁差、章动、极移、UT1-UTC 和折射改正，通常由专业软件链统一处理，本书此处不展开算法细节。
</div>


对于常规连通阵观测，`ICRS/J2000` 目录位置往往足以表达我们关心的几何起点；但对高精度 VLBI、脉冲星定时或航天器跟踪而言，参考架和历元就不能只当作标签，而必须真正进入延迟模型和相关处理。换句话说，本节先建立的是*概念清晰度*，而不是把全部天体测量修正逐一推导完。


### 示例：把射电源目录位置变成可计算的数值

下面用几个常见亮射电源做一个小例子。这里不依赖专门天文学软件，而是直接把目录中常见的时分秒、度分秒表示转换成可用于计算的小时数、角度和弧度。这样做的目的，是让“源目录条目”与“后续几何计算中的数组变量”之间的关系变得直观。


In [ ]:
radio_sources = [
    ('Cas A',   (23, 23, 24.0), (+1, 58, 48, 54.0)),
    ('Cyg A',   (19, 59, 28.3), (+1, 40, 44,  2.1)),
    ('Tau A',   ( 5, 34, 31.9), (+1, 22,  0, 52.2)),
    ('Vir A',   (12, 30, 49.4), (+1, 12, 23, 28.0)),
    ('Sgr A*',  (17, 45, 40.0), (-1, 29,  0, 28.0)),
    ('Pictor A',( 5, 19, 49.7), (-1, 45, 46, 44.0)),
    ('3C286',   (13, 31,  8.3), (+1, 30, 30, 33.0)),
]


def hms_to_hours(h, m, s):
    return h + m / 60.0 + s / 3600.0


def signed_dms(sign, d, m, s):
    return sign * (abs(d) + m / 60.0 + s / 3600.0)


names = []
ra_hours = []
dec_deg = []
for name, (h, m, s), (sign, d, amin, asec) in radio_sources:
    names.append(name)
    ra_hours.append(hms_to_hours(h, m, s))
    dec_deg.append(signed_dms(sign, d, amin, asec))

ra_hours = np.array(ra_hours)
dec_deg = np.array(dec_deg)
ra_rad = np.deg2rad(ra_hours * 15.0)
dec_rad = np.deg2rad(dec_deg)

for name, ra_h, dec_d in zip(names, ra_hours, dec_deg):
    print(f'{name:8s}  RA = {ra_h:6.3f} h ({ra_h * 15:7.3f} deg)   Dec = {dec_d:+8.3f} deg')

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.scatter(ra_hours, dec_deg, s=80, color='tab:red')
for name, x, y in zip(names, ra_hours, dec_deg):
    ax.text(x + 0.18, y + 2.0, name, fontsize=10)

ax.set_xlim(24, 0)
ax.set_ylim(-90, 90)
ax.set_xticks(np.arange(24, -0.1, -2))
ax.set_yticks(np.arange(-90, 91, 30))
ax.grid(alpha=0.3)
ax.set_xlabel('赤经 [h]')
ax.set_ylabel('赤纬 [deg]')
ax.set_title('若干亮射电源的目录位置（示意）')
plt.show()


这个例子看似简单，却恰好对应了实际观测中最常见的第一步：把源表中的赤经赤纬读入程序，统一转换到弧度，再交给后续的时角、地平坐标和成像几何模块处理。真正的调度软件会在此基础上继续调用更完整的参考架与历元变换，但其输入本质上仍然来自这样的目录坐标。


#### 本节要点

* 赤道坐标是射电干涉观测链条的出发点，目录位置、校准源和相位中心都依赖它；
* 赤经常用小时表示，赤纬常用角度表示，但公式和代码最好统一转成弧度；
* `ICRS/J2000`、`B1950` 等标签不是装饰，而是目录位置能否正确使用的前提；
* 观测软件最终会把目录位置继续转换为与观测时刻、台站位置相关的量，这正是下一节时角与地方恒星时要处理的内容。


***

下一节：[3.2 时角、地方恒星时与过中天](3_2_hour_angle.ipynb)
